# 01 - Dataset Exploration (Weeks 1-2)

**Objective:** Understand the structure and content of the CRMLS Sold datasets before any analysis.

Data: monthly `CRMLSSoldYYYYMM.csv` files (Dec 2025 - May 2026) from the CoreLogic Trestle API.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append('../src')
from config import RAW

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

## 1. List the monthly files

In [ ]:
files = sorted(RAW.glob('CRMLSSold*.csv'))
files

## 2. Inspect a single month

The handbook workflow: `read_csv` -> `head` -> `columns` -> `describe`.

In [ ]:
df = pd.read_csv(files[0], low_memory=False)
print('shape:', df.shape)
df.head()

In [ ]:
list(df.columns)

## 3. Per-file overview

Row counts and the CloseDate range for each month - confirms each file holds exactly one month.

In [ ]:
frames = []
for f in files:
    d = pd.read_csv(f, low_memory=False)
    d['SourceMonth'] = f.stem.replace('CRMLSSold', '')
    cd = pd.to_datetime(d['CloseDate'], errors='coerce')
    print(f'{f.name:24s} rows={len(d):6d}  CloseDate {cd.min().date()} -> {cd.max().date()}')
    frames.append(d)

sold = pd.concat(frames, ignore_index=True)
print('\nCombined shape:', sold.shape)

## 4. Property types

Only `Residential` sale records belong in this analysis. Leases are rentals (their 'price' is rent),
and Land/Commercial/Income are different markets.

In [ ]:
sold['PropertyType'].value_counts(dropna=False)

## 5. Price distribution: all types vs Residential

Mixing leases into ClosePrice corrupts the distribution. Compare the two.

In [ ]:
print('All types:')
print(sold['ClosePrice'].describe())
print('\nResidential only:')
res = sold[sold['PropertyType'] == 'Residential']
print(res['ClosePrice'].describe())

## 6. Completeness of key fields (Residential)

In [ ]:
key = ['ClosePrice', 'OriginalListPrice', 'ListPrice', 'LivingArea', 'CloseDate',
       'DaysOnMarket', 'City', 'PostalCode', 'CountyOrParish', 'ListAgentFullName',
       'ListOfficeName', 'BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt',
       'Latitude', 'Longitude']
(res[key].isna().mean() * 100).round(2).sort_values(ascending=False)

## Findings

- 6 months of data (Dec 2025 - May 2026), one month per file, **124,404 rows x 79 columns** combined.
- **Residential = 82,643 rows** is the analysis target. Leases (29,673) must be excluded - they corrupt price stats.
- Key analytic fields are **<0.2% missing** on the Residential subset - excellent quality.
- Raw `ClosePrice` still spans **$2 to $796M** even within Residential -> needs IQR outlier removal (Week 9).

**Next:** Weeks 3-4 - aggregate all months into one combined dataset (`sold_combined.csv`).